# 1. 대회의 심층 분석

## 1.1 대회 목적

이 대회는 미국 상장기업의 SEC 공시자료(10-K, 10-Q)에서 추출한 펀더멘털 데이터를 이용해 각 기업-분기 관측치의 **1년 후 주식 수익률(`return_pct`)** 을 예측하는 회귀 문제이다.

학습 데이터는 2019년부터 2022년까지의 기업-분기 관측치로 구성되어 있고, 평가 데이터는 2024년 관측치로 구성되어 있다. 각 행은 특정 기업이 특정 분기 시점에 가지고 있던 재무 상태, 수익성, 성장성, 밸류에이션 정보를 나타낸다.

이 대회가 어려운 이유는 주식 수익률의 변경이 심하고, 극단적인 상승과 하락이 존재하는 fat-tailed 분포를 가지기 때문이다. 따라서 모든 종목의 수익률을 정확히 맞추는 것보다, 제한적인 펀더멘털 신호를 안정적으로 찾아내고 극단 오차를 줄이는 것이 중요하다.

## 1.2 평가 지표 및 평가 방식

이 대회의 공식 평가 지표는 **RMSE(Root Mean Squared Error)** 이다. 예측 대상은 `return_pct`이며, 모델이 예측한 1년 후 수익률과 실제 1년 후 수익률 사이의 RMSE가 낮을수록 좋은 점수를 받는다.

RMSE는 다음과 같이 계산된다.

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}
$$

여기서  
- $y_i$는 실제 1년 후 수익률이다.
- $\hat{y}_i$는 모델이 예측한 1년 후 수익률이다.
- $n$은 평가 데이터의 행 개수이다.

`return_pct`는 퍼센트 단위의 수익률이므로, RMSE도 퍼센트 포인트 단위로 해석할 수 있다. 예를 들어 RMSE가 40이라면 예측값이 실제 수익률과 평균적으로 약 40%p 수준의 오차를 가진다는 의미로 볼 수 있다.

> **AI 활용 질문**: “RMSE로 평가하는 회귀 대회에서 예측값의 극단값이 왜 문제가 되는지 설명해줘.”
>
> **이 질문을 사용한 이유**: 이 대회는 수익률 극단값이 많기 때문에, RMSE가 모델 전략에 어떤 영향을 주는지 먼저 이해할 필요가 있었다.

## 1.3 제출 제한 및 대회 규정의 특징

> **AI 활용 질문**: “Kaggle 대회에서 하루 제출 제한과 private leaderboard 기준이 모델 선택에 어떤 영향을 주는지 정리해줘.”
>
> **이 질문을 사용한 이유**: public 점수만 보고 반복 제출하는 방식이 위험한지 판단하고, 내부 검증 중심의 제출 전략을 세우기 위해 사용했다.

이 대회의 주요 제출 제한과 규정은 다음과 같다.

| 구분 | 내용 |
|---|---|
| 1일 제출 제한 | 하루 최대 5회 제출 가능 |
| 최종 제출 선택 | 최대 2개 Final Submission 선택 가능 |
| 팀 제한 | 최대 5명 |
| 최종 순위 기준 | Private Leaderboard 기준 |
| 상금 | 1등 $1,000 |
| 데이터 사용 범위 | Competition Use 및 Non-Commercial & Academic Research |
| 외부 데이터 사용 | 공개 접근 가능하고 합리적 비용이면 사용 가능 |
| 우승자 의무 | 최종 모델 코드와 재현 가능한 설명 제출 필요 |
| 우승 솔루션 라이선스 | Open Source 조건 |

이 중 모델링 전략에 가장 직접적으로 영향을 주는 것은 **1일 제출 제한**과 **Private Leaderboard 기준 최종 평가**이다. 하루에 5번만 제출할 수 있으므로 public leaderboard 점수만 보고 반복적으로 제출값을 조정하는 방식은 위험하다.

따라서 제출 전 내부 validation을 먼저 신뢰할 수 있게 만들고, public leaderboard는 최종 확인용으로 사용하는 것이 적절하다.

## 1.4 리더보드의 표출 방식

Kaggle 리더보드는 일반적으로 Public Leaderboard와 Private Leaderboard로 나뉜다.

| 리더보드 | 의미 | 특징 |
|---|---|---|
| Public Leaderboard | 테스트 데이터 일부에 대한 점수 | 대회 진행 중 공개 |
| Private Leaderboard | 숨겨진 테스트 데이터에 대한 점수 | 최종 순위 결정 |
| 최종 평가 | Private Leaderboard 기준 | public 점수와 순위가 달라질 수 있음 |

Public Leaderboard는 모델 개선 방향을 확인하는 데 도움이 되지만, 최종 성능을 완전히 보장하지 않는다. 특히 이 대회처럼 수익률 분포가 불안정하고 극단값이 많은 문제에서는 public test subset에 우연히 잘 맞은 모델이 private test에서는 성능이 떨어질 수 있다.

따라서 이 대회에서는 public score 자체보다 다음 관계를 함께 해석해야 한다.

| 비교 항목 | 해석 |
|---|---|
| validation 개선 + public 개선 | 전략이 비교적 신뢰 가능 |
| validation 개선 + public 악화 | 검증 방식이 test 분포를 반영하지 못했을 가능성 |
| validation 악화 + public 개선 | public leaderboard 과적합 가능성 |
| public만 보고 반복 개선 | private leaderboard에서 성능 하락 위험 |

결론적으로 public leaderboard는 참고 지표이고, 최종 판단은 시간 기반 validation과 private leaderboard 일반화 가능성을 중심으로 해야 한다.

## 1.5 대회의 특별한 규정과 분석상 주의점

이 대회에는 분석과 모델링 과정에서 주의해야 할 규정과 데이터 구조가 있다.

첫째, train과 test 사이에 의도적인 시간 간격이 있다. 학습 데이터는 2019~2022년 관측치이고, 테스트 데이터는 2024년 관측치이다. 이는 시간 누수를 방지하기 위한 설계이다. 따라서 검증 방식도 미래 예측 상황과 비슷하게 구성해야 한다.

둘째, test 데이터의 실제 `return_pct`는 공개되지 않는다. 참가자는 test row별 예측값만 제출하고, 점수는 Kaggle이 숨겨진 실제값과 비교해 계산한다.

셋째, 여러 분기에서 같은 ticker가 반복해서 등장할 수 있다. 따라서 랜덤 분할을 사용할 경우 같은 기업의 다른 시점 정보가 train과 validation에 동시에 들어갈 수 있다. 이 경우 실제 미래 예측보다 검증 점수가 과도하게 좋게 나올 수 있다.

넷째, 외부 데이터는 사용할 수 있지만 공개적으로 접근 가능하고 합리적인 비용이어야 한다. 이 노트북에서는 과제의 재현성과 공정성을 위해 기본 제공 데이터만 사용한다.


# 2. 학습/평가 데이터 심층 분석

## 2.1 데이터 구조와 도메인 이해

> **AI 활용 질문**: “기업 펀더멘털로 1년 후 주식 수익률을 예측할 때 일반적인 정형 회귀 문제와 다른 점은 무엇이야?”
>
> **이 질문을 사용한 이유**: 이 데이터를 단순 회귀 데이터가 아니라 금융 데이터로 해석해야 하는 이유를 정리하기 위해 사용했다.

이 데이터는 미국 상장기업의 SEC 공시자료에서 추출한 재무·펀더멘털 데이터이다. 각 행은 하나의 기업이 특정 분기 시점에 가지고 있던 재무 상태를 나타내며, 학습 데이터에는 그 시점 이후 1년 동안의 주가 수익률인 `return_pct`가 포함되어 있다.

즉, 이 문제는 단순히 재무제표 숫자를 예측하는 문제가 아니라, **기업의 현재 펀더멘털 상태가 향후 1년 주가 성과와 어떤 관계를 가지는지 찾는 문제**이다.

이 대회의 핵심 난점은 다음과 같다.

| 난점 | 의미 |
|---|---|
| 수익률의 잡음 | 주가는 기업 재무뿐 아니라 금리, 경기, 투자심리, 섹터 순환, 뉴스에 영향을 받는다. |
| fat-tailed target | 일부 종목은 1년 동안 수백 % 상승하거나 크게 하락할 수 있다. |
| 시간 차이 | train은 2019~2022년, test는 2024년이므로 시장 국면이 다르다. |
| 같은 기업의 반복 관측 | 한 기업이 여러 분기 관측치로 등장하므로 검증 방식에 주의해야 한다. |
| 결측값의 의미 | 재무 데이터의 결측은 단순 오류가 아니라 기업 특성 자체일 수 있다. |

따라서 이 데이터는 “모든 종목을 정확히 맞추는 문제”라기보다, **극단값과 시장 국면 변화 속에서도 일반화 가능한 약한 신호를 찾는 문제**로 이해해야 한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

display(train.head())
display(test.head())

## 2.2 데이터 파일의 역할

이 대회에서 제공되는 파일은 세 개이다.

| 파일 | 역할 |
|---|---|
| `train.csv` | 2019~2022년 기업-분기 관측치와 정답 `return_pct` 포함 |
| `test.csv` | 2024년 기업-분기 관측치, 정답 없음 |
| `sample_submission.csv` | 제출 파일 형식 예시 |

학습 데이터에는 `period_start`, `period_end`, `return_pct`가 존재하지만, 테스트 데이터에는 `return_pct`가 없다. 또한 test 데이터는 2024년 예측 대상이므로 미래 데이터에 대한 일반화 성능이 중요하다.

특히 주의할 점은 train의 ticker는 실제 종목 코드인 반면, test의 ticker는 `stock_0000` 형태로 익명화되어 있다는 점이다. 따라서 test에서는 특정 기업명을 직접 이용한 암기식 예측이 불가능하다. 모델은 개별 ticker 이름이 아니라 재무지표 자체의 패턴을 학습해야 한다.

In [ ]:
print("train columns:")
print(train.columns.tolist())

print("\ntest columns:")
print(test.columns.tolist())

print("\ntrain에는 있지만 test에는 없는 컬럼:")
print(sorted(set(train.columns) - set(test.columns)))

print("\ntest에는 있지만 train에는 없는 컬럼:")
print(sorted(set(test.columns) - set(train.columns)))

print("\ntrain 기간:")
print(train["start_year"].value_counts().sort_index())

print("\ntest 기간:")
print(test["start_year"].value_counts().sort_index())

print("\ntrain ticker 예시:", train["ticker"].head(10).tolist())
print("test ticker 예시:", test["ticker"].head(10).tolist())

ticker_overlap = len(set(train["ticker"]) & set(test["ticker"]))
print("\ntrain/test ticker 이름 겹침 개수:", ticker_overlap)

## 2.3 컬럼 의미의 한국어 해석

> **AI 활용 질문**: “P/E, P/B, P/S, ROE, ROA, debt_to_equity 같은 재무지표를 주식 수익률 예측 관점에서 쉽게 설명해줘.”
>
> **이 질문을 사용한 이유**: 컬럼명을 단순 번역하는 것이 아니라, 각 지표가 기업의 가치평가·수익성·재무위험을 어떻게 나타내는지 이해하기 위해 사용했다.

아래 표는 각 컬럼의 의미를 실제 분석 관점에서 해석한 것이다. 이 대회에서는 단순히 컬럼명을 아는 것보다, 각 지표가 어떤 기업 특성을 나타내는지 이해하는 것이 중요하다.

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `id` | 행 식별자 | 제출 파일에서 test row와 예측값을 연결하는 키 |
| `ticker` | 종목 코드 | train은 실제 티커, test는 익명화된 종목명 |
| `start_year` | 관측 시작 연도 | 학습/검증을 시간 기준으로 나눌 때 핵심 컬럼 |
| `period_start` | 수익률 측정 시작일 | train에서 1년 보유기간의 시작일 |
| `period_end` | 수익률 측정 종료일 | train에서 1년 보유기간의 종료일 |
| `return_pct` | 1년 후 수익률 | 예측해야 하는 target |

### Valuation: 기업이 비싼지 싼지

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `market_cap` | 시가총액 | 기업 규모. 대형주는 안정적, 소형주는 변동성이 클 수 있음 |
| `pe_ttm` | 최근 12개월 주가수익비율 | 이익 대비 주가가 비싼지 판단. 매우 크거나 음수이면 해석 주의 |
| `price_to_book` | 주가순자산비율 | 장부가치 대비 시장 평가. 낮으면 저평가 후보일 수 있음 |
| `price_to_sales` | 주가매출비율 | 매출 대비 주가 수준. 적자 기업 분석에 P/E보다 유용할 수 있음 |
| `growth_pe_ratio` | 성장률 조정 P/E | 성장 대비 가격 수준. PEG와 비슷한 개념 |

### Profitability: 기업이 돈을 잘 버는지

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `gross_margin` | 매출총이익률 | 제품/서비스 자체의 수익성 |
| `operating_margin` | 영업이익률 | 본업에서 얼마나 효율적으로 돈을 버는지 |
| `net_margin` | 순이익률 | 모든 비용을 반영한 최종 수익성 |
| `roa` | 총자산이익률 | 자산을 얼마나 효율적으로 이익으로 바꾸는지 |
| `roe` | 자기자본이익률 | 주주자본 대비 이익 창출력 |
| `rote` | 유형자기자본이익률 | 무형자산을 제외한 자본 대비 수익성 |

### Growth: 기업이 성장하고 있는지

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `revenue_growth_3y` | 3년 매출 성장률 | 장기 성장 추세 |
| `revenue_growth_yoy` | 전년 대비 매출 성장률 | 최근 성장 속도 |

### Income: 손익 규모

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `revenue_ttm` | 최근 12개월 매출 | 기업의 매출 규모 |
| `net_income_ttm` | 최근 12개월 순이익 | 최종 이익 규모 |
| `income_before_tax` | 세전이익 | 세금 전 이익 창출력 |
| `eps_basic` | 기본 주당순이익 | 보통주 1주당 이익 |
| `eps_diluted` | 희석 주당순이익 | 전환증권 등을 고려한 보수적 EPS |

### Balance Sheet: 재무 안정성

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `total_assets` | 총자산 | 기업이 보유한 전체 자원 |
| `stockholders_equity` | 주주자본 | 자산에서 부채를 뺀 주주 몫 |
| `current_assets` | 유동자산 | 1년 내 현금화 가능한 자산 |
| `current_liabilities` | 유동부채 | 1년 내 갚아야 할 부채 |
| `long_term_debt` | 장기부채 | 장기 차입 부담 |
| `goodwill` | 영업권 | 인수합병 등으로 발생한 무형가치 |
| `inventory` | 재고자산 | 제조/유통 기업의 운영 상태를 보여줌 |

### Ratios: 유동성과 부채 부담

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `current_ratio` | 유동비율 | 단기 지급능력 |
| `quick_ratio` | 당좌비율 | 재고를 제외한 더 보수적인 단기 지급능력 |
| `debt_to_equity` | 부채자본비율 | 자본 대비 부채 부담 |

### Dividends: 배당

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `dividend_yield` | 배당수익률 | 주가 대비 배당 수준 |
| `dividends_ttm` | 최근 12개월 주당 배당 | 주당 배당 규모 |
| `dividends_paid_ttm` | 최근 12개월 총 배당금 | 기업 전체가 지급한 배당 규모 |

### Other: 기타 구조적 정보

| 컬럼 | 한국어 의미 | 실제 해석 |
|---|---|---|
| `shares_outstanding` | 발행주식수 | 기업 규모와 EPS 계산에 영향 |
| `shares_diluted` | 희석주식수 | 잠재 주식까지 고려한 주식 수 |
| `sector_code` | 섹터 코드 | 업종 차이 반영. 같은 재무지표라도 업종별 의미가 다름 |

## 2.4 컬럼 그룹화 전략

이 대회에서 모든 컬럼을 동일하게 다루면 해석력이 떨어진다. 재무지표는 성격별로 묶어서 봐야 한다.

특히 중요한 관점은 다음과 같다.

| 그룹 | 핵심 질문 | 모델링 아이디어 |
|---|---|---|
| Valuation | 지금 주가가 비싼가 싼가? | 저평가 지표와 미래 수익률의 관계 확인 |
| Profitability | 기업이 돈을 잘 버는가? | 수익성이 높은 기업의 안정성 확인 |
| Growth | 성장하고 있는가? | 성장주 프리미엄 또는 과대평가 여부 확인 |
| Balance Sheet | 재무구조가 안전한가? | 부채 부담과 하락 위험 확인 |
| Liquidity | 단기 지급능력이 있는가? | 재무위험 신호 확인 |
| Dividends | 주주환원 성향이 있는가? | 가치주/성숙기업 특성 반영 |
| Sector | 어떤 업종인가? | 업종별 재무지표 기준 차이 보정 |

중요한 점은 같은 숫자라도 업종에 따라 의미가 다르다는 것이다. 예를 들어, 높은 `price_to_sales`는 기술주에서는 흔할 수 있지만 유틸리티나 금융업에서는 다르게 해석된다. 따라서 `sector_code`는 단순 범주형 변수가 아니라, 재무지표 해석 기준을 바꾸는 중요한 변수이다.

In [ ]:
feature_groups = {
    "valuation": ["pe_ttm", "price_to_book", "price_to_sales", "growth_pe_ratio"],
    "profitability": ["gross_margin", "operating_margin", "net_margin", "roa", "roe", "rote"],
    "growth": ["revenue_growth_3y", "revenue_growth_yoy"],
    "income": ["revenue_ttm", "net_income_ttm", "income_before_tax", "eps_basic", "eps_diluted"],
    "balance_sheet": [
        "total_assets", "stockholders_equity", "current_assets", "current_liabilities",
        "long_term_debt", "goodwill", "inventory"
    ],
    "ratios": ["current_ratio", "quick_ratio", "debt_to_equity"],
    "dividends": ["dividend_yield", "dividends_ttm", "dividends_paid_ttm"],
    "other": ["shares_outstanding", "shares_diluted", "sector_code"]
}

for group, cols in feature_groups.items():
    existing_cols = [c for c in cols if c in train.columns]
    print(f"{group}: {len(existing_cols)}개")
    print(existing_cols)
    print()

## 2.5 Target `return_pct` 심층 분석

> **AI 활용 질문**: “주식 수익률 target이 fat-tailed 분포를 가질 때 어떤 통계량과 그래프를 확인해야 해?”
>
> **이 질문을 사용한 이유**: 평균과 표준편차만으로는 수익률 분포를 제대로 설명하기 어렵기 때문에, 분위수·극단값·clipping 전후 분포를 함께 확인하기 위해 사용했다.

`return_pct`는 관측 시점 이후 1년 동안의 주가 수익률이다. 이 대회의 평가 지표가 RMSE이므로 target 분포 분석은 매우 중요하다.

주식 수익률은 일반적으로 정규분포보다 꼬리가 두껍다. 일부 종목은 매우 큰 상승률을 기록하고, 일부 종목은 거의 전액 손실에 가까운 하락률을 기록할 수 있다. 이러한 극단값은 RMSE를 크게 흔들 수 있다.

따라서 target 분석에서는 평균보다 다음 값들이 더 중요하다.

| 확인 항목 | 이유 |
|---|---|
| 중앙값 | 일반적인 종목의 대표 수익률 |
| 5%, 95% 분위수 | 보통의 하락/상승 범위 |
| 1%, 99% 분위수 | 극단값의 경계 |
| 최솟값/최댓값 | RMSE를 크게 흔들 수 있는 관측치 |
| 연도별 target 분포 | 시장 국면 변화 확인 |

특히 연도별 target 분포를 보면, 특정 연도에는 시장 전체가 상승하거나 하락했을 수 있다. 이 경우 랜덤 검증은 시장 국면을 섞어버려 실제 2024년 예측 상황을 제대로 반영하지 못할 수 있다.

In [ ]:
target_summary = train["return_pct"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

display(target_summary)

print("100% 초과 상승 비율:", (train["return_pct"] > 100).mean())
print("-50% 미만 하락 비율:", (train["return_pct"] < -50).mean())
print("300% 초과 상승 비율:", (train["return_pct"] > 300).mean())

year_target = train.groupby("start_year")["return_pct"].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
).round(2)

display(year_target)

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(train["return_pct"], bins=100)
plt.title("Target Distribution: return_pct")
plt.xlabel("1-Year Forward Return (%)")
plt.ylabel("Count")
plt.show()

lower, upper = train["return_pct"].quantile([0.01, 0.99])

plt.figure(figsize=(10, 5))
plt.hist(train["return_pct"].clip(lower, upper), bins=100)
plt.title("Target Distribution after 1%~99% Clipping")
plt.xlabel("Clipped 1-Year Forward Return (%)")
plt.ylabel("Count")
plt.show()

print("1% quantile:", lower)
print("99% quantile:", upper)

## 2.6 Target 분석 결과 해석

`return_pct`의 전체 분포를 확인한 결과, 이 대회가 일반적인 회귀 문제보다 훨씬 어려운 이유가 명확하게 나타났다.

전체 학습 데이터의 `return_pct` 평균은 약 **18.78%**이지만, 중앙값은 **3.50%**에 불과하다. 평균이 중앙값보다 훨씬 큰 이유는 일부 종목이 매우 큰 상승률을 기록하면서 전체 평균을 끌어올렸기 때문이다. 즉, target 분포는 오른쪽 꼬리가 매우 긴 형태이다.

분위수 기준으로 보면 1% 지점은 약 **-80.20%**, 5% 지점은 약 **-56.20%**이다. 이는 일부 종목은 1년 동안 절반 이상 하락했으며, 하위 1%는 거의 전액 손실에 가까운 수준까지 하락했음을 의미한다. 반대로 95% 지점은 약 **118.08%**, 99% 지점은 약 **299.17%**이다. 즉 상위 5% 종목은 1년 동안 두 배 이상 상승했고, 상위 1% 종목은 약 네 배 가까운 수익률을 기록했다.

가장 중요한 점은 최댓값이 **10,571.11%**로 매우 크다는 것이다. 이는 일반적인 주식 수익률 범위를 크게 벗어난 초극단값이며, RMSE에 매우 큰 영향을 줄 수 있다. RMSE는 오차를 제곱하기 때문에 이런 극단 관측치를 제대로 맞추지 못하면 전체 점수가 크게 악화된다.

연도별로 보면 2020년의 평균 수익률은 **73.87%**, 중앙값은 **42.79%**로 다른 연도보다 압도적으로 높다. 이는 2020년 이후 시장 반등 구간의 영향이 강하게 반영된 것으로 볼 수 있다. 반면 2021년은 평균 **-10.51%**, 중앙값 **-12.16%**로 전체적으로 부진한 수익률을 보인다. 2019년과 2022년은 평균과 중앙값이 상대적으로 낮고 안정적이다.

이 결과는 검증 전략에 중요한 시사점을 준다. 랜덤 split을 사용하면 2020년의 강한 상승장과 2021년의 부진한 시장이 학습/검증에 섞이게 된다. 그러면 실제 2024년 test처럼 미래 특정 시점에 대해 예측하는 상황을 제대로 반영하지 못할 수 있다. 따라서 이 대회에서는 시간 기반 검증이 필요하다.

또한 원본 histogram은 대부분의 관측치가 좁은 구간에 몰려 있고, 극단적으로 큰 오른쪽 꼬리 때문에 전체 분포를 제대로 보기 어렵다. 1%~99% clipping 후 histogram을 보면 대부분의 일반적인 수익률 분포가 훨씬 명확해진다. 이때도 분포는 완전한 정규분포가 아니라 오른쪽 꼬리가 긴 비대칭 구조를 보인다.

따라서 이후 모델링에서는 다음 전략이 필요하다.

1. target의 극단값을 그대로 학습할지, clipping할지 비교한다.
2. 예측값이 비정상적으로 커지는 것을 막기 위해 prediction clipping을 검토한다.
3. 2020년 같은 특수한 시장 국면이 모델을 과도하게 지배하지 않도록 시간 기반 검증을 사용한다.
4. RMSE는 극단값에 민감하므로, 단일 모델보다 안정적인 앙상블 전략을 고려한다.
5. 평균 수익률보다 중앙값과 분위수를 함께 보며 예측값 분포를 관리한다.

결론적으로, 이 target은 “평균적인 수익률을 맞추는 문제”가 아니라 **극단적인 수익률과 시장 국면 변화 속에서 안정적인 예측을 만드는 문제**이다.

## 2.7 결측치 분석: 결측은 단순한 빈칸이 아니다

> **AI 활용 질문**: “재무 데이터에서 결측값이 단순 오류가 아니라 기업 특성일 수 있는 예시를 알려줘.”
>
> **이 질문을 사용한 이유**: 결측값을 무조건 제거하거나 평균으로 대체하기보다, 배당 없음·재고 없음·적자 기업 같은 의미 있는 신호로 해석하기 위해 사용했다.

재무 데이터에서 결측값은 단순히 데이터가 누락된 것이 아니라, 기업의 특성을 나타내는 신호일 수 있다.

예를 들어 배당 관련 컬럼이 비어 있거나 0에 가까운 기업은 배당을 하지 않는 성장기업일 수 있다. `inventory`가 비어 있거나 작게 나타나는 기업은 재고가 거의 필요 없는 소프트웨어, 금융, 서비스 기업일 수 있다. `pe_ttm`이 결측인 기업은 최근 12개월 이익이 음수라 P/E 계산이 불가능한 적자 기업일 가능성이 있다.

따라서 결측값을 단순히 평균이나 중앙값으로 채우는 것만으로는 부족하다. 결측 자체가 기업의 재무 상태, 업종, 성장 단계, 위험도를 반영할 수 있기 때문이다.

이번 분석에서는 다음을 확인한다.

1. train과 test에서 결측률이 높은 컬럼은 무엇인가?
2. train과 test의 결측 패턴이 비슷한가?
3. 결측률 차이가 큰 컬럼은 test 일반화에 위험한가?
4. 모델링에서 결측 여부 indicator를 추가할 필요가 있는가?

In [ ]:
missing = pd.DataFrame({
    "train_missing_count": train.isna().sum(),
    "train_missing_rate": train.isna().mean(),
    "test_missing_count": test.isna().sum(),
    "test_missing_rate": test.isna().mean()
})

missing["missing_rate_diff"] = missing["test_missing_rate"] - missing["train_missing_rate"]
missing = missing.sort_values("train_missing_rate", ascending=False)

display(missing.head(25))

print("train 결측이 있는 컬럼 수:", (train.isna().sum() > 0).sum())
print("test 결측이 있는 컬럼 수:", (test.isna().sum() > 0).sum())

print("\ntrain 결측률 20% 이상 컬럼:")
display(missing[missing["train_missing_rate"] >= 0.20])

print("\ntest 결측률이 train보다 5%p 이상 높은 컬럼:")
display(missing[missing["missing_rate_diff"] >= 0.05].sort_values("missing_rate_diff", ascending=False))

plt.figure(figsize=(12, 5))
missing.head(15)[["train_missing_rate", "test_missing_rate"]].plot(kind="bar", figsize=(12, 5))
plt.title("Top 15 Missing Rate Features: Train vs Test")
plt.xlabel("Feature")
plt.ylabel("Missing Rate")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 2.8 결측치 분석 결과 해석

결측치 분석 결과, 이 데이터는 결측이 매우 많은 재무 데이터라는 점이 확인되었다. train과 test 모두 결측이 있는 컬럼 수는 **33개**로 동일하다. 즉, 결측은 일부 컬럼에서 우연히 발생한 문제가 아니라 데이터 전반에 존재하는 구조적 특징이다.

가장 결측률이 높은 컬럼은 `dividends_paid_ttm`이다. train 결측률은 약 **93.2%**, test 결측률은 약 **94.5%**이다. `dividend_yield`와 `dividends_ttm`도 각각 train에서 약 **72.0%**, **70.8%**의 결측률을 보인다. 이는 대부분의 기업이 배당 관련 정보가 없거나, 배당을 하지 않는 기업이 많다는 뜻으로 해석할 수 있다. 특히 성장기업이나 적자기업은 배당을 하지 않는 경우가 많기 때문에, 배당 컬럼의 결측은 단순한 누락이 아니라 기업의 성장 단계나 주주환원 정책을 나타내는 신호일 수 있다.

`gross_margin`의 결측률도 train 약 **62.5%**, test 약 **61.1%**로 매우 높다. 이는 모든 업종에서 매출총이익률이 같은 방식으로 계산되지 않기 때문일 수 있다. 예를 들어 금융업은 제조업처럼 매출원가와 매출총이익 구조를 해석하기 어렵다. 따라서 `gross_margin` 결측은 업종 구조와 관련된 정보일 가능성이 있다.

`inventory`도 train 약 **49.9%**, test 약 **55.7%**가 결측이다. 재고자산은 제조업, 유통업에서는 중요하지만 소프트웨어, 금융, 서비스 기업에서는 거의 의미가 없거나 작게 나타날 수 있다. 따라서 `inventory` 결측은 해당 기업이 재고 중심 비즈니스인지 아닌지를 구분하는 신호가 될 수 있다.

또 하나 중요한 점은 test에서 결측률이 train보다 높아진 컬럼이 많다는 것이다. `long_term_debt`, `roa`, `goodwill`, `total_assets`, `stockholders_equity`, `rote`, `pe_ttm`, `price_to_sales`, `growth_pe_ratio` 등은 test 결측률이 train보다 5%p 이상 높다. 이는 2024년 test 데이터가 2019~2022년 train 데이터와 동일한 결측 구조를 갖지 않는다는 의미이다.

특히 `roa`, `roe`, `rote`, `pe_ttm`, `price_to_book`, `price_to_sales`처럼 수익성·밸류에이션 핵심 지표의 결측률이 test에서 높아진 점은 중요하다. 이 컬럼들은 모델이 수익률을 예측할 때 사용할 가능성이 높은 핵심 변수인데, test에서 더 자주 비어 있다면 단순한 중앙값 대체만으로는 일반화 성능이 떨어질 수 있다.

따라서 결측치 처리 전략은 다음과 같이 정리한다.

1. 결측값은 단순히 제거하지 않는다. 결측률이 높은 컬럼이 많아 행이나 컬럼을 삭제하면 정보 손실이 크다.
2. 수치형 결측값은 중앙값 등으로 대체하되, 결측 여부를 나타내는 indicator feature를 추가한다.
3. 배당, 재고, 이익률, 부채 관련 결측은 기업 특성 또는 업종 특성으로 해석한다.
4. train/test 결측률 차이가 큰 컬럼은 분포 이동 가능성이 있으므로 모델 해석과 검증에서 주의한다.
5. 결측을 자연스럽게 처리할 수 있는 tree-based model이나 결측 indicator를 포함한 선형/부스팅 모델을 고려한다.

결론적으로 이 데이터에서 결측값은 단순한 전처리 문제가 아니라, 기업의 비즈니스 모델·업종·재무 상태·데이터 시점 차이를 반영하는 중요한 정보이다.

## 2.9 컬럼별 이상치와 왜도 분석

> **AI 활용 질문**: “P/E, ROE, debt_to_equity 같은 재무비율에서 극단값이 생기는 이유와 처리 방법을 알려줘.”
>
> **이 질문을 사용한 이유**: 극단값을 단순 오류로 삭제하지 않고, 분모가 작거나 음수인 재무비율의 특성을 이해한 뒤 clipping 필요성을 판단하기 위해 사용했다.

재무 데이터는 대부분 정규분포를 따르지 않는다. 기업 규모, 매출, 자산, 부채 같은 금액 변수는 소수의 초대형 기업 때문에 오른쪽 꼬리가 매우 길어질 수 있다. 또한 P/E, P/B, ROE, 부채비율 같은 비율 변수는 분모가 작거나 음수일 때 비정상적으로 큰 값이 발생할 수 있다.

이상치와 왜도 분석이 중요한 이유는 다음과 같다.

| 분석 항목 | 필요한 이유 |
|---|---|
| 최솟값과 최댓값 | 비정상적으로 큰 값이 모델을 흔드는지 확인 |
| 1%, 99% 분위수 | 일반적인 값의 범위를 파악 |
| 평균과 중앙값 차이 | 소수 극단값이 평균을 왜곡하는지 확인 |
| 왜도(skewness) | 분포가 한쪽으로 얼마나 치우쳤는지 확인 |
| train/test 극단값 차이 | test에 train보다 더 극단적인 값이 있는지 확인 |

특히 RMSE 기반 대회에서는 target뿐 아니라 feature의 극단값도 중요하다. 극단적인 feature 값이 있으면 모델이 특정 관측치에 과도하게 반응하여 예측값이 비정상적으로 커질 수 있다.

따라서 이번 단계에서는 각 수치형 feature의 분포 특성을 정리하고, clipping 또는 log 변환이 필요한 후보를 찾는다.

In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()

# id와 target은 feature 분석에서 제외
feature_cols = [c for c in numeric_cols if c not in ["id", "return_pct"]]

feature_profile = []

for col in feature_cols:
    tr = train[col].replace([np.inf, -np.inf], np.nan)
    te = test[col].replace([np.inf, -np.inf], np.nan) if col in test.columns else pd.Series(dtype=float)
    
    feature_profile.append({
        "feature": col,
        "train_missing_rate": tr.isna().mean(),
        "test_missing_rate": te.isna().mean(),
        "train_mean": tr.mean(),
        "train_median": tr.median(),
        "train_std": tr.std(),
        "train_min": tr.min(),
        "train_p01": tr.quantile(0.01),
        "train_p99": tr.quantile(0.99),
        "train_max": tr.max(),
        "train_skew": tr.skew(),
        "test_min": te.min(),
        "test_p01": te.quantile(0.01),
        "test_p99": te.quantile(0.99),
        "test_max": te.max(),
        "test_skew": te.skew()
    })

feature_profile = pd.DataFrame(feature_profile)
feature_profile["abs_train_skew"] = feature_profile["train_skew"].abs()
feature_profile["mean_median_gap"] = (feature_profile["train_mean"] - feature_profile["train_median"]).abs()

print("왜도가 큰 컬럼 Top 15")
display(
    feature_profile
    .sort_values("abs_train_skew", ascending=False)
    .head(15)
    .round(3)
)

print("평균-중앙값 차이가 큰 컬럼 Top 15")
display(
    feature_profile
    .sort_values("mean_median_gap", ascending=False)
    .head(15)
    .round(3)
)

print("train 99% 분위수 대비 최댓값이 과도하게 큰 컬럼 Top 15")
feature_profile["max_to_p99_ratio"] = (
    feature_profile["train_max"].abs() / feature_profile["train_p99"].replace(0, np.nan).abs()
)

display(
    feature_profile
    .sort_values("max_to_p99_ratio", ascending=False)
    .head(15)
    .round(3)
)

## 2.10 컬럼별 이상치와 왜도 분석 결과 해석

컬럼별 분포를 확인한 결과, 이 데이터의 수치형 feature들은 대부분 정규분포와 거리가 멀고 극단값의 영향을 크게 받는 것으로 나타났다. 이는 재무 데이터에서 흔히 나타나는 특징이다. 기업 규모, 매출, 자산처럼 금액 단위가 큰 변수는 소수의 초대형 기업 때문에 오른쪽 꼬리가 길어지고, P/E, P/B, ROE 같은 비율 변수는 분모가 작거나 음수일 때 비정상적으로 큰 값이 발생할 수 있다.

가장 먼저 눈에 띄는 것은 `revenue_growth_yoy`, `revenue_growth_3y`, `eps_basic`, `eps_diluted`, `roa`, `roe`, `rote`, `price_to_book`, `pe_ttm`, `price_to_sales`, `debt_to_equity` 등 주요 재무비율 컬럼의 왜도가 매우 크다는 점이다. 이 컬럼들은 평균과 중앙값의 차이도 크고, 표준편차도 일반적인 범위를 크게 벗어난다. 즉, 대부분의 기업은 비교적 좁은 범위에 몰려 있지만 일부 기업의 값이 매우 크게 튀는 구조이다.

특히 `pe_ttm`은 중앙값이 약 15 수준인데 평균은 수백만 단위로 나타난다. 이는 일반적인 기업의 P/E 수준과 평균값이 완전히 다른 의미를 가진다는 뜻이다. 일부 비정상적으로 큰 P/E 또는 음수 P/E가 평균을 왜곡하고 있으므로, `pe_ttm`은 원본 값을 그대로 사용하기보다 clipping 또는 변환이 필요하다.

`price_to_book`, `roe`, `rote`, `debt_to_equity`도 비슷한 문제가 있다. 이 지표들은 자본이 매우 작거나 음수인 기업에서 극단적인 값이 발생할 수 있다. 예를 들어 ROE는 순이익을 자기자본으로 나눈 값이므로, 자기자본이 작거나 음수이면 값이 비정상적으로 커질 수 있다. 따라서 단순히 “ROE가 높을수록 좋은 기업”이라고 해석하면 안 된다.

`eps_basic`, `eps_diluted`도 평균과 중앙값 차이가 크다. EPS는 주당순이익을 의미하지만, 주식 수나 일회성 손익에 따라 극단값이 발생할 수 있다. 이 경우 모델이 EPS 극단값을 과도하게 신호로 받아들일 위험이 있다.

금액 규모 변수인 `total_assets`, `revenue_ttm`, `long_term_debt`, `stockholders_equity`, `current_assets`, `current_liabilities`, `goodwill`, `inventory`는 평균이 중앙값보다 훨씬 크다. 이는 소수의 초대형 기업이 평균을 끌어올리는 구조이다. 이런 변수는 원본 스케일보다 `log1p` 변환을 적용하는 것이 더 적절하다.

또한 `dividends_paid_ttm`은 중앙값이 0인데 평균은 큰 값을 가진다. 이는 대부분의 기업은 배당금 지급이 없거나 결측이지만, 일부 대형 배당 기업이 매우 큰 배당금을 지급하기 때문이다. 이런 컬럼은 “값의 크기”뿐 아니라 “배당을 하는지 여부” 자체가 중요한 정보가 될 수 있다.

이 분석을 바탕으로 모델링 전략은 다음과 같이 정리할 수 있다.

1. 규모 변수는 `log1p` 변환을 적용한다.
   - 예: `market_cap`, `revenue_ttm`, `total_assets`, `current_assets`, `long_term_debt`, `shares_outstanding`

2. 비율 변수는 극단값 clipping을 적용한다.
   - 예: `pe_ttm`, `price_to_book`, `price_to_sales`, `roe`, `rote`, `debt_to_equity`, `growth_pe_ratio`

3. 결측률이 높은 변수는 결측 indicator를 추가한다.
   - 예: 배당, 재고, 부채, 성장률, 수익성 관련 컬럼

4. 평균보다 중앙값과 분위수를 더 신뢰한다.
   - 평균은 극단값에 크게 왜곡되어 일반적인 기업을 대표하지 못한다.

5. 선형 모델에는 스케일링과 변환이 필수이고, tree-based model에도 clipping을 적용하면 예측 안정성이 좋아질 수 있다.

결론적으로 이 데이터는 원본 feature를 그대로 사용하는 것보다, 결측 indicator, clipping, log 변환을 적용한 후 모델링하는 것이 더 적절하다.

## 2.11 핵심 feature 분포 시각화

앞선 표에서 모든 수치형 feature에 극단값과 왜도가 존재한다는 점을 확인했다. 하지만 표만으로는 분포의 형태를 직관적으로 이해하기 어렵다.

따라서 이번 단계에서는 모델링에 중요한 대표 feature들을 선택해 원본 분포와 변환 후 분포를 비교한다.

선택한 feature는 다음과 같다.

| feature | 선택 이유 |
|---|---|
| `pe_ttm` | 대표적인 P/E 밸류에이션 지표이며 극단값이 매우 큼 |
| `price_to_book` | 자본 대비 주가 수준을 보여주지만 음수·극단값 가능 |
| `price_to_sales` | 적자 기업에도 사용 가능한 valuation 지표 |
| `roe` | 수익성 지표지만 자기자본이 작거나 음수이면 극단값 발생 |
| `debt_to_equity` | 재무위험 지표지만 자본이 작으면 비정상적으로 커질 수 있음 |
| `revenue_growth_yoy` | 최근 성장률이며 일부 기업의 폭발적 성장률 존재 |
| `total_assets` | 기업 규모를 나타내며 초대형 기업 때문에 오른쪽 꼬리가 김 |
| `revenue_ttm` | 매출 규모를 나타내며 log 변환 후보 |

이 분석은 단순 시각화가 아니라, 이후 전처리 전략을 결정하기 위한 근거이다.

In [ ]:
important_features = [
    "pe_ttm",
    "price_to_book",
    "price_to_sales",
    "roe",
    "debt_to_equity",
    "revenue_growth_yoy",
    "total_assets",
    "revenue_ttm"
]

important_features = [c for c in important_features if c in train.columns]

for col in important_features:
    s = train[col].replace([np.inf, -np.inf], np.nan).dropna()
    
    q01 = s.quantile(0.01)
    q99 = s.quantile(0.99)
    s_clip = s.clip(q01, q99)
    
    plt.figure(figsize=(10, 4))
    plt.hist(s_clip, bins=60)
    plt.title(f"{col} Distribution after 1%~99% Clipping")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()
    
    print(f"{col}")
    print("  missing rate:", round(train[col].isna().mean(), 4))
    print("  median:", round(s.median(), 4))
    print("  mean:", round(s.mean(), 4))
    print("  1%:", round(q01, 4))
    print("  99%:", round(q99, 4))
    print("  min:", round(s.min(), 4))
    print("  max:", round(s.max(), 4))
    print("-" * 80)

## 2.12 핵심 feature 분포 시각화 결과 해석

대표 feature들의 분포를 시각화한 결과, 이 데이터의 재무지표들은 원본값을 그대로 사용하기 어렵다는 점이 명확하게 나타났다. 대부분의 관측치는 좁은 구간에 몰려 있지만, 일부 기업의 값이 극단적으로 크거나 작아 전체 분포를 왜곡하고 있다.

먼저 `pe_ttm`은 중앙값이 약 **15.21**로 일반적인 P/E 수준에 가깝지만, 평균은 약 **4,419,551**로 비정상적으로 크다. 1% 분위수는 약 **-3,785**, 99% 분위수는 약 **1,073**이며, 원본 최솟값은 약 **-237,899,082**, 최댓값은 약 **12,515,525,649**이다. 이는 P/E가 기업의 이익이 0에 가깝거나 음수일 때 폭발적으로 커질 수 있기 때문이다. 따라서 `pe_ttm`은 평균을 신뢰하기 어렵고, clipping이 반드시 필요한 대표적인 feature이다.

`price_to_book`도 중앙값은 약 **2.96**이지만 평균은 약 **1,318.64**로 크게 왜곡되어 있다. 99% 분위수도 약 **1,028.78**로 매우 높고, 원본 최댓값은 **5,271,275.1**까지 존재한다. P/B는 장부가치 대비 주가 수준을 나타내지만, 자기자본이 작거나 음수에 가까운 기업에서는 매우 큰 값 또는 음수 값이 나올 수 있다. 따라서 이 컬럼 역시 단순히 “낮으면 저평가, 높으면 고평가”로 해석하기 어렵다.

`price_to_sales`는 중앙값이 약 **2.65**로 비교적 현실적인 수준이지만 평균은 약 **38.47**이고, 최댓값은 **53,118.76**이다. P/S는 적자기업에도 사용할 수 있어 유용하지만, 매출이 매우 작거나 시장 기대가 과도하게 반영된 기업에서는 극단값이 발생한다. 따라서 이 컬럼도 clipping을 통해 과도한 영향을 줄여야 한다.

`roe`는 중앙값이 약 **10.25**이지만 평균은 약 **1,205.66**이다. 최솟값은 약 **-1,206,823.77**, 최댓값은 약 **3,361,580.43**으로 매우 극단적이다. ROE는 순이익을 자기자본으로 나눈 값이므로, 자기자본이 작거나 음수이면 숫자가 비정상적으로 커질 수 있다. 따라서 ROE가 극단적으로 높다고 해서 반드시 좋은 기업이라고 해석하면 안 된다.

`debt_to_equity`는 중앙값이 약 **0.65**로 일반적으로 해석 가능한 수준이지만, 평균은 약 **31.81**이고 최댓값은 약 **69,786.5**이다. 부채자본비율은 자기자본이 작을 때 크게 튀므로, 극단값은 재무위험 신호일 수도 있지만 계산상의 불안정성일 수도 있다.

`revenue_growth_yoy`는 중앙값이 약 **8.2%**, 평균은 약 **63.73%**이다. 최댓값은 **414,794.44%**로 매우 크다. 이는 매출 기준이 작았던 기업이 일시적으로 큰 성장률을 기록했거나, 특수한 회계·기업 이벤트가 반영되었을 가능성이 있다. 성장률 feature도 극단값을 그대로 사용하면 모델이 일부 기업에 과도하게 반응할 수 있다.

반면 `total_assets`와 `revenue_ttm` 같은 규모 변수는 값의 단위 자체가 매우 크다. `total_assets`의 중앙값은 약 **56.4억 달러**, 평균은 약 **404.6억 달러**이고, 최댓값은 약 **4.3조 달러**이다. `revenue_ttm`도 중앙값은 약 **24.8억 달러**, 평균은 약 **113.4억 달러**, 최댓값은 약 **6,001억 달러**이다. 이는 소수의 초대형 기업이 평균을 크게 끌어올리는 구조이다. 이런 변수는 clipping뿐 아니라 `log1p` 변환을 적용해야 모델이 규모 차이를 안정적으로 학습할 수 있다.

이 분석을 통해 다음 전처리 방향이 명확해졌다.

| feature 유형 | 문제점 | 전처리 방향 |
|---|---|---|
| P/E, P/B, P/S | 분모가 작거나 음수일 때 극단값 발생 | 1%~99% 또는 더 강한 clipping |
| ROE, ROA, ROTE | 자본·자산이 작거나 음수일 때 극단값 발생 | clipping + 결측 indicator |
| debt_to_equity | 자기자본이 작으면 폭발 | clipping + 부채 존재 여부 feature |
| revenue growth | 작은 기준값 때문에 성장률 폭발 | clipping |
| 매출·자산·부채 규모 변수 | 초대형 기업 때문에 오른쪽 꼬리 | log1p 변환 |
| 배당 변수 | 대부분 0 또는 결측, 일부 기업만 큰 값 | 배당 여부 indicator + log/clipping |

결론적으로, 이 대회의 feature는 원본값을 그대로 사용하면 모델이 극단값에 과도하게 끌려갈 위험이 크다. 이후 모델링에서는 **결측 indicator 추가, 수치형 feature clipping, 규모 변수 log 변환**을 기본 전처리 전략으로 사용한다.

## 2.13 섹터별 분석: 같은 재무지표도 업종에 따라 의미가 다르다

`sector_code`는 기업이 속한 업종을 나타내는 컬럼이다. 이 컬럼은 단순한 범주형 변수가 아니라, 재무지표를 해석하는 기준 역할을 한다.

예를 들어 같은 P/E 30이라도 기술주는 성장 기대가 높기 때문에 자연스러울 수 있지만, 유틸리티 기업에서는 비싼 평가로 해석될 수 있다. 또한 금융업은 일반 제조업과 자산·부채 구조가 다르기 때문에 `debt_to_equity`, `total_assets`, `current_ratio` 같은 지표를 동일한 기준으로 비교하기 어렵다.

따라서 섹터 분석에서는 다음을 확인한다.

1. train과 test의 섹터 비중이 비슷한가?
2. 섹터별 `return_pct` 평균과 중앙값이 다른가?
3. 어떤 섹터가 극단 수익률을 많이 포함하는가?
4. 섹터별 차이가 크다면, 모델에 `sector_code`를 반드시 포함해야 하는가?
5. 더 나아가 섹터 내 상대 순위 feature를 만들 필요가 있는가?

이 분석은 이후 모델 개선 전략에서 **섹터 내 valuation rank**, **섹터 내 profitability rank** 같은 파생변수를 만들 근거가 된다.

In [ ]:
sector_map = {
    0: "Technology",
    1: "Financial Services",
    2: "Industrials",
    3: "Healthcare",
    4: "Consumer Cyclical",
    5: "Real Estate",
    6: "Basic Materials",
    7: "Consumer Defensive",
    8: "Energy",
    9: "Utilities",
    10: "Communication Services"
}

train["sector_name"] = train["sector_code"].map(sector_map)
test["sector_name"] = test["sector_code"].map(sector_map)

sector_target = train.groupby("sector_name")["return_pct"].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    q05=lambda x: x.quantile(0.05),
    q95=lambda x: x.quantile(0.95),
    max="max"
).sort_values("median", ascending=False).round(2)

display(sector_target)

sector_dist = pd.DataFrame({
    "train_count": train["sector_name"].value_counts(),
    "test_count": test["sector_name"].value_counts(),
    "train_rate": train["sector_name"].value_counts(normalize=True),
    "test_rate": test["sector_name"].value_counts(normalize=True)
}).fillna(0)

sector_dist["rate_diff"] = sector_dist["test_rate"] - sector_dist["train_rate"]
sector_dist = sector_dist.sort_values("rate_diff", ascending=False)

display(sector_dist.round(4))

plt.figure(figsize=(12, 5))
sector_target["median"].sort_values().plot(kind="barh")
plt.title("Median return_pct by Sector")
plt.xlabel("Median 1-Year Return (%)")
plt.ylabel("Sector")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sector_dist[["train_rate", "test_rate"]].sort_values("train_rate").plot(kind="barh", figsize=(12, 5))
plt.title("Sector Distribution: Train vs Test")
plt.xlabel("Ratio")
plt.ylabel("Sector")
plt.tight_layout()
plt.show()

## 2.14 섹터별 분석 결과 해석

섹터별 `return_pct`를 비교한 결과, 업종에 따라 1년 후 수익률 분포가 뚜렷하게 다르게 나타났다.

중앙값 기준으로 가장 높은 섹터는 **Energy**이며, 중앙값 수익률은 약 **18.49%**이다. 그다음은 **Technology 8.60%**, **Industrials 7.64%**, **Basic Materials 7.28%** 순이다. 반면 **Real Estate -5.08%**, **Communication Services -4.32%**, **Utilities -1.43%**, **Financial Services -0.66%**는 중앙값이 음수이다.

이 결과는 같은 시장 기간 안에서도 섹터별 수익률 구조가 크게 다르다는 것을 보여준다. 따라서 `sector_code`는 단순한 부가 정보가 아니라, 미래 수익률 예측에 중요한 기준 변수로 사용해야 한다.

특히 이 대회에서는 같은 재무지표라도 섹터에 따라 해석이 달라진다. 예를 들어 높은 P/E는 기술주에서는 성장 기대를 반영한 자연스러운 값일 수 있지만, 유틸리티나 부동산 섹터에서는 과대평가 신호일 수 있다. 또한 금융업은 일반 제조업과 자산·부채 구조가 다르기 때문에 `debt_to_equity`, `current_ratio`, `total_assets` 같은 지표를 동일한 기준으로 비교하기 어렵다.

섹터별 극단값도 다르게 나타난다. Industrials, Consumer Cyclical, Financial Services, Healthcare 등은 매우 큰 최댓값을 가지고 있어 특정 종목의 급등이 섹터 평균을 크게 끌어올렸을 가능성이 있다. 따라서 섹터별 평균보다는 중앙값과 분위수를 함께 보는 것이 더 안정적이다.

train/test 섹터 비중을 비교하면 전체적으로 큰 차이는 없다. 가장 큰 차이도 Healthcare가 test에서 약 1.48%p 증가하고, Real Estate가 약 0.88%p 감소한 정도이다. 즉, test 데이터가 특정 섹터에 과도하게 치우쳐 있지는 않다. 이는 train에서 학습한 섹터 효과가 test에서도 어느 정도 유지될 가능성을 보여준다.

이 분석에서 얻은 모델링 아이디어는 다음과 같다.

1. `sector_code`는 반드시 feature로 사용한다.
2. 전체 시장 기준의 절대값뿐 아니라 섹터 내 상대 순위 feature를 만들 수 있다.
3. valuation 지표는 섹터별 기준으로 비교하는 것이 더 적절하다.
4. profitability와 leverage 지표도 섹터별로 해석을 달리해야 한다.
5. 섹터별 수익률 차이가 있으므로, 섹터별 target encoding이나 섹터별 rank feature를 검토할 수 있다.

결론적으로 이 데이터는 전체 기업을 하나의 기준으로 비교하는 것보다, **섹터 안에서 상대적으로 싼 기업, 수익성이 좋은 기업, 성장성이 높은 기업을 찾는 방향**이 더 적절하다.

## 2.15 학습데이터와 평가데이터의 feature 분포 비교

> **AI 활용 질문**: “train은 2019~2022년이고 test는 2024년일 때, 어떤 방식으로 feature 분포 차이를 비교해야 해?”
>
> **이 질문을 사용한 이유**: 단순 평균 비교만으로는 부족하므로, 중앙값 차이·결측률 차이·표준화 평균 차이를 함께 확인하기 위해 사용했다.

이 대회에서 train/test 분포 비교는 매우 중요하다. train은 2019~2022년 데이터이고, test는 2024년 데이터이므로 두 데이터는 시간적으로 분리되어 있다. 따라서 test는 단순히 train에서 랜덤 추출된 데이터가 아니라, 미래 시점의 새로운 시장 국면을 반영할 수 있다.

이미 앞에서 결측률과 섹터 비중을 비교한 결과, 결측률은 여러 핵심 feature에서 test가 더 높았고, 섹터 비중은 비교적 안정적인 것으로 확인되었다.

이번 단계에서는 수치형 feature 전체를 대상으로 train/test 분포 차이를 비교한다. 단순 평균 차이만 보면 스케일이 큰 컬럼이 과도하게 중요해 보일 수 있으므로, 표준화된 평균 차이와 중앙값 차이를 함께 확인한다.

확인할 항목은 다음과 같다.

| 항목 | 의미 |
|---|---|
| train/test 평균 차이 | 전체 수준 변화 확인 |
| train/test 중앙값 차이 | 일반적인 기업 기준 변화 확인 |
| 결측률 차이 | 데이터 수집 또는 기업 구성 변화 확인 |
| 표준화된 평균 차이 | 스케일이 다른 feature 간 비교 |
| 분위수 차이 | 극단값 영향을 줄인 분포 변화 확인 |

이 분석의 목적은 모델이 train에서 학습한 패턴을 2024년 test에도 안정적으로 적용할 수 있는지 판단하는 것이다.

In [ ]:
common_numeric_features = [
    c for c in test.select_dtypes(include=[np.number]).columns
    if c in train.columns and c not in ["id"]
]

drift_rows = []

for col in common_numeric_features:
    tr = train[col].replace([np.inf, -np.inf], np.nan)
    te = test[col].replace([np.inf, -np.inf], np.nan)
    
    pooled_std = tr.std()
    if pd.isna(pooled_std) or pooled_std == 0:
        standardized_mean_diff = np.nan
    else:
        standardized_mean_diff = abs(te.mean() - tr.mean()) / pooled_std
    
    median_denom = abs(tr.median()) + 1e-9
    relative_median_diff = abs(te.median() - tr.median()) / median_denom
    
    drift_rows.append({
        "feature": col,
        "train_mean": tr.mean(),
        "test_mean": te.mean(),
        "train_median": tr.median(),
        "test_median": te.median(),
        "train_missing": tr.isna().mean(),
        "test_missing": te.isna().mean(),
        "missing_diff": te.isna().mean() - tr.isna().mean(),
        "standardized_mean_diff": standardized_mean_diff,
        "relative_median_diff": relative_median_diff
    })

drift = pd.DataFrame(drift_rows)

print("표준화 평균 차이가 큰 feature Top 15")
display(
    drift
    .sort_values("standardized_mean_diff", ascending=False)
    .head(15)
    .round(4)
)

print("중앙값 차이가 큰 feature Top 15")
display(
    drift
    .sort_values("relative_median_diff", ascending=False)
    .head(15)
    .round(4)
)

print("결측률 차이가 큰 feature Top 15")
display(
    drift
    .assign(abs_missing_diff=lambda x: x["missing_diff"].abs())
    .sort_values("abs_missing_diff", ascending=False)
    .head(15)
    .round(4)
)

## 2.16 학습데이터와 평가데이터의 feature 분포 비교 결과 해석

train/test feature 분포를 비교한 결과, test 데이터는 train 데이터와 완전히 같은 분포라고 보기 어렵다. 특히 일부 수익성, 밸류에이션, 성장성, 결측 패턴에서 차이가 나타난다.

먼저 `start_year`는 train과 test를 가장 명확하게 구분하는 컬럼이다. train은 2019~2022년, test는 2024년이므로 `start_year`의 차이가 가장 크게 나타나는 것은 당연하다. 그러나 이 컬럼은 test의 모든 값이 2024이고 train에는 2024가 없기 때문에, 모델 feature로 직접 사용하는 것은 조심해야 한다. `start_year`는 예측 feature라기보다 검증 데이터를 시간 기준으로 나누기 위한 컬럼으로 사용하는 것이 더 적절하다.

표준화 평균 차이를 보면 `roa`, `rote`, `roe`, `net_margin`, `operating_margin`, `pe_ttm`, `price_to_book`, `debt_to_equity` 등이 상위에 나타난다. 이들은 대부분 수익성, 자기자본 수익률, 밸류에이션, 부채비율 관련 지표이다. 다만 이 컬럼들은 앞선 분석에서 이미 극단값이 매우 큰 것으로 확인되었기 때문에, 평균 차이는 실제 일반적인 기업의 차이라기보다 일부 극단값의 영향일 가능성이 크다. 따라서 평균 차이만 보고 test 분포가 완전히 다르다고 판단하면 안 되고, 중앙값도 함께 봐야 한다.

중앙값 기준으로 보면 `revenue_growth_yoy`, `revenue_growth_3y`, `growth_pe_ratio`, `dividend_yield`, `dividends_ttm`, `net_margin`, `current_assets`, `total_assets`, `goodwill` 등에서 차이가 나타난다. 특히 `revenue_growth_yoy`는 train 중앙값이 약 8.2인데 test 중앙값은 약 5.48로 낮아졌다. 이는 2024년 test 기업들의 최근 매출 성장성이 train 기간보다 다소 낮게 나타날 수 있음을 의미한다. 성장률은 미래 수익률 예측에 중요한 feature이므로 이 차이는 모델 일반화에 영향을 줄 수 있다.

결측률 차이에서는 `income_before_tax`가 train보다 test에서 결측률이 크게 낮아진 반면, `long_term_debt`, `roa`, `goodwill`, `total_assets`, `stockholders_equity`, `rote`, `pe_ttm`, `price_to_sales`, `growth_pe_ratio`, `revenue_growth_3y` 등은 test에서 결측률이 더 높다. 이는 test 데이터가 단순히 train에서 랜덤하게 뽑힌 것이 아니라, 2024년의 새로운 데이터 구조와 결측 패턴을 가지고 있음을 보여준다.

중요한 점은 섹터 비중은 train/test 간 큰 차이가 없었지만, 개별 feature의 결측률과 분포는 차이가 있다는 것이다. 즉, test는 업종 구성 자체가 크게 바뀐 데이터라기보다는, 같은 업종 안에서도 2024년의 재무 상태와 데이터 가용성이 달라진 데이터라고 볼 수 있다.

이 결과는 모델링에 다음과 같은 시사점을 준다.

1. `start_year`는 모델 입력 feature로 직접 쓰기보다 시간 기반 검증 분할에 사용하는 것이 적절하다.
2. 평균 차이가 큰 feature는 극단값의 영향을 받을 수 있으므로 clipping 후 다시 비교하는 것이 필요하다.
3. test에서 결측률이 높아진 feature가 많으므로 결측 indicator를 포함해야 한다.
4. 성장성 지표의 중앙값 차이는 2024년 시장 환경이 train 기간과 다를 수 있음을 보여준다.
5. 섹터 비중은 안정적이므로 섹터 효과는 유지하되, 섹터 내 상대 순위 feature를 만드는 것이 유효할 수 있다.
6. train/test 분포 차이가 존재하므로 랜덤 validation만으로는 leaderboard 성능을 예측하기 어렵다.

결론적으로 이 대회에서는 test가 train과 완전히 동일한 분포라고 가정하면 안 된다. 따라서 모델 개선은 단순 학습 성능보다 **시간 기반 검증, 결측 패턴 대응, 극단값 clipping, 섹터 내 상대 비교**를 중심으로 진행해야 한다.

## 2.17 검증데이터 구축 방안

> **AI 활용 질문**: “미래 연도 데이터를 예측하는 금융 데이터에서 랜덤 검증보다 시간 기반 검증이 필요한 이유를 설명해줘.”
>
> **이 질문을 사용한 이유**: test가 2024년 데이터이므로, 랜덤 split이 실제 평가 상황을 과도하게 낙관적으로 만들 수 있는지 판단하기 위해 사용했다.

이 대회에서는 검증 데이터를 어떻게 나누는지가 모델 성능 평가의 핵심이다. 일반적인 정형 데이터 문제에서는 랜덤 train/validation split을 자주 사용하지만, 이 대회에서는 랜덤 분할만으로는 실제 test 상황을 제대로 반영하기 어렵다.

그 이유는 다음과 같다.

| 이유 | 설명 |
|---|---|
| 시간 구조 존재 | train은 2019~2022년, test는 2024년 데이터이다. |
| 시장 국면 차이 | 2020년은 강한 상승장, 2021년은 부진한 구간으로 target 분포가 다르다. |
| 같은 ticker 반복 관측 | 같은 기업이 여러 분기 등장하므로 랜덤 split 시 기업 정보가 섞일 수 있다. |
| test는 미래 시점 | 실제 예측 대상은 과거 데이터가 아니라 2024년 미래 데이터이다. |
| feature drift 존재 | test에서 결측률과 일부 feature 분포가 train과 다르다. |

따라서 이 대회에서 랜덤 split은 baseline 확인용으로만 사용하고, 모델 선택과 전략 판단에는 시간 기반 검증을 우선해야 한다.

## 2.18 검증 전략 후보

이 대회에서 사용할 수 있는 검증 전략은 다음과 같다.

| 검증 방식 | 설명 | 장점 | 한계 |
|---|---|---|---|
| Random Holdout | 전체 train에서 무작위로 validation 분리 | 간단하고 빠름 | 시간 구조와 ticker 반복 문제를 반영하지 못함 |
| KFold | train을 여러 fold로 나누어 평균 성능 계산 | 안정적인 평균 점수 확보 | 미래 예측 상황과 다름 |
| GroupKFold by ticker | 같은 ticker가 train/valid에 동시에 들어가지 않게 분리 | 종목 암기 방지 | 시간 구조를 반영하지 못함 |
| Time-based Holdout | 과거 연도로 학습하고 최근 연도로 검증 | test의 미래 예측 상황과 유사 | validation이 특정 연도 하나에 의존 |
| Rolling Validation | 여러 연도를 순차적으로 validation으로 사용 | 연도별 시장 국면 변화 확인 가능 | 데이터가 적은 초기 연도는 학습량 부족 |

이 대회의 기본 검증 전략은 **Time-based Holdout**으로 설정한다.

가장 현실적인 기본 설정은 다음과 같다.

- train: 2019~2021년
- validation: 2022년

이 방식은 2024년 test를 직접 볼 수 없는 상황에서, 가장 최근 train 연도인 2022년을 미래 검증 구간처럼 사용하는 방법이다.

In [ ]:
# 시간 기반 검증 데이터 구축

train_part = train[train["start_year"] <= 2021].copy()
valid_part = train[train["start_year"] == 2022].copy()

print("Time-based Holdout")
print("train_part shape:", train_part.shape)
print("valid_part shape:", valid_part.shape)

print("\ntrain_part year distribution:")
display(train_part["start_year"].value_counts().sort_index())

print("\nvalid_part year distribution:")
display(valid_part["start_year"].value_counts().sort_index())

print("\nTarget summary: train_part")
display(train_part["return_pct"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(2))

print("\nTarget summary: valid_part")
display(valid_part["return_pct"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(2))

# ticker overlap 확인
train_tickers = set(train_part["ticker"])
valid_tickers = set(valid_part["ticker"])

print("\ntrain_part ticker 수:", len(train_tickers))
print("valid_part ticker 수:", len(valid_tickers))
print("train/valid ticker 겹침 수:", len(train_tickers & valid_tickers))
print("valid ticker 중 train에도 있었던 비율:", len(train_tickers & valid_tickers) / len(valid_tickers))

## 2.19 Time-based Holdout 검증 데이터 구축 결과 해석

시간 기반 검증 데이터를 `2019~2021년 학습`, `2022년 검증`으로 나눈 결과, 학습 구간은 **16,436개 행**, 검증 구간은 **6,634개 행**으로 구성되었다. 이 방식은 test가 2024년 데이터라는 점을 고려할 때, 랜덤 분할보다 실제 미래 예측 상황에 더 가깝다.

target 분포를 비교하면 학습 구간과 검증 구간 사이에 차이가 존재한다. 학습 구간의 `return_pct` 평균은 약 **21.37%**, 표준편차는 약 **158.96**이다. 반면 2022년 검증 구간의 평균은 약 **12.36%**, 표준편차는 약 **64.77**이다. 중앙값은 학습 구간 **3.64%**, 검증 구간 **3.26%**로 비슷하지만, 평균과 표준편차는 학습 구간이 훨씬 크다.

이 차이는 학습 구간에 2020년의 강한 상승장과 초극단 수익률이 포함되어 있기 때문이다. 학습 구간의 최댓값은 **10,571.11%**인 반면, 검증 구간의 최댓값은 **1,932.83%**이다. 또한 99% 분위수도 학습 구간은 약 **327.94%**, 검증 구간은 약 **213.69%**로 차이가 난다. 즉, 2022년 검증 데이터는 학습 구간보다 극단적인 상승 관측치가 적고 변동성이 낮은 구조이다.

이 결과는 모델링에서 중요한 시사점을 준다. 학습 데이터 전체를 그대로 사용하면 모델이 2020년과 같은 강한 상승장 및 초극단값에 과도하게 영향을 받을 수 있다. 하지만 실제 검증 구간인 2022년은 상대적으로 평균 수익률과 변동성이 낮기 때문에, 극단값을 과하게 따라가는 모델은 검증 성능이 나빠질 수 있다.

ticker overlap도 중요하다. 검증 구간의 ticker 중 약 **93.5%**가 학습 구간에도 존재한다. 이는 2022년 holdout이 완전히 새로운 기업을 예측하는 문제라기보다는, 과거에 관측된 기업들의 미래 시점 수익률을 예측하는 구조에 가깝다는 의미이다. 따라서 이 검증 방식은 test의 시간 구조를 어느 정도 반영하지만, 종목 단위의 완전한 일반화 성능을 평가하지는 못한다.

따라서 검증 전략은 다음과 같이 해석한다.

1. 기본 모델 선택은 `2019~2021 train / 2022 validation` 기준으로 수행한다.
2. 이 방식은 랜덤 split보다 test의 미래 예측 상황에 더 가깝다.
3. 다만 ticker가 대부분 겹치므로 종목 암기 가능성을 완전히 제거하지는 못한다.
4. 학습 구간에는 2020년의 초강세장이 포함되어 있어 target clipping 필요성이 크다.
5. 2022년 validation 하나만으로 모든 시장 국면을 대표할 수 없으므로 rolling validation도 함께 확인한다.

결론적으로 Time-based Holdout은 이 대회의 기본 검증 전략으로 적절하지만, 단일 검증 점수만 절대적으로 신뢰하기보다는 연도별 rolling validation과 함께 해석해야 한다.

## 2.20 Rolling Validation 필요성

2022년을 검증 구간으로 사용하는 Time-based Holdout은 test의 미래 예측 상황을 어느 정도 반영한다. 그러나 한 개 연도만 검증에 사용하면 특정 시장 국면에 과도하게 의존할 수 있다.

앞선 target 분석에서 2020년은 평균과 중앙값이 매우 높은 상승장 성격을 보였고, 2021년은 평균과 중앙값이 음수인 부진한 구간으로 나타났다. 따라서 어떤 연도를 validation으로 선택하느냐에 따라 모델 성능과 최적 전략이 달라질 수 있다.

이를 보완하기 위해 rolling validation 관점을 함께 사용한다. rolling validation은 과거 연도들로 학습하고 다음 연도를 검증하는 방식이다.

예를 들어:

| 학습 구간 | 검증 구간 |
|---|---|
| 2019 | 2020 |
| 2019~2020 | 2021 |
| 2019~2021 | 2022 |

이 방식은 시장 국면이 달라질 때 모델이 얼마나 안정적으로 작동하는지 확인하는 데 도움이 된다.

In [ ]:
rolling_summary = []

years = sorted(train["start_year"].unique())

for valid_year in years:
    train_years = [y for y in years if y < valid_year]
    
    if len(train_years) == 0:
        continue
    
    tr = train[train["start_year"].isin(train_years)]
    va = train[train["start_year"] == valid_year]
    
    train_tickers = set(tr["ticker"])
    valid_tickers = set(va["ticker"])
    overlap_ratio = len(train_tickers & valid_tickers) / len(valid_tickers)
    
    rolling_summary.append({
        "train_years": f"{min(train_years)}~{max(train_years)}" if len(train_years) > 1 else str(train_years[0]),
        "valid_year": valid_year,
        "train_rows": len(tr),
        "valid_rows": len(va),
        "train_target_mean": tr["return_pct"].mean(),
        "valid_target_mean": va["return_pct"].mean(),
        "train_target_median": tr["return_pct"].median(),
        "valid_target_median": va["return_pct"].median(),
        "valid_target_std": va["return_pct"].std(),
        "valid_target_p95": va["return_pct"].quantile(0.95),
        "valid_target_p99": va["return_pct"].quantile(0.99),
        "valid_target_max": va["return_pct"].max(),
        "ticker_overlap_ratio": overlap_ratio
    })

rolling_summary = pd.DataFrame(rolling_summary).round(2)
display(rolling_summary)

## 2.21 Rolling Validation 결과 해석

Rolling validation 결과, 연도별 시장 국면이 매우 다르다는 점이 다시 확인되었다. 이는 이 대회에서 단순 랜덤 검증이 위험한 이유를 잘 보여준다.

첫 번째 검증 구간인 2020년은 매우 강한 상승장 성격을 가진다. 2019년 데이터로 학습하고 2020년을 검증하면, validation target 평균은 **73.87%**, 중앙값은 **42.79%**, 표준편차는 **254.15**이다. 또한 99% 분위수는 **505.66%**, 최댓값은 **10,571.11%**로 매우 극단적이다. 즉, 2020년은 일반적인 1년 수익률 분포라기보다 코로나 이후 강한 반등과 일부 종목의 폭발적 상승이 반영된 특수한 구간으로 볼 수 있다.

두 번째 검증 구간인 2021년은 반대로 부진한 시장 국면을 보인다. 2019~2020년으로 학습하고 2021년을 검증하면, validation target 평균은 **-10.51%**, 중앙값은 **-12.16%**이다. 95% 분위수도 **48.69%**, 99% 분위수도 **113.63%**로 2020년에 비해 훨씬 낮다. 이 구간에서는 무리하게 높은 수익률을 예측하는 모델이 오히려 성능을 크게 악화시킬 가능성이 있다.

세 번째 검증 구간인 2022년은 2020년보다는 훨씬 안정적이고, 2021년보다는 양호한 구간이다. 2019~2021년으로 학습하고 2022년을 검증하면, validation target 평균은 **12.36%**, 중앙값은 **3.26%**, 표준편차는 **64.77**이다. 99% 분위수는 **213.69%**, 최댓값은 **1,932.83%**로 여전히 극단값은 존재하지만, 2020년보다는 훨씬 완화된 분포이다.

ticker overlap 비율은 모든 rolling validation에서 높게 나타난다. 2020년 validation은 약 **92%**, 2021년은 약 **88%**, 2022년은 약 **94%**의 validation ticker가 이전 학습 구간에도 존재한다. 이는 train 내부 검증이 완전히 새로운 기업을 예측하는 문제라기보다, 기존에 관측된 기업들의 다른 시점 수익률을 예측하는 구조에 가깝다는 것을 의미한다.

이 결과에서 얻은 핵심 결론은 다음과 같다.

1. 연도별 target 분포가 크게 달라서, 하나의 validation 점수만으로 모델을 평가하면 위험하다.
2. 2020년은 극단 상승장이므로 이 구간에 과도하게 맞춘 모델은 다른 연도에서 성능이 나빠질 수 있다.
3. 2021년은 부진한 구간이므로 보수적인 예측과 prediction clipping의 필요성을 보여준다.
4. 2022년은 test와 시간적으로 가장 가까운 검증 구간이므로 최종 모델 선택의 기준으로 가장 적절하다.
5. ticker overlap이 높으므로, 검증 점수가 좋아도 완전히 새로운 종목에 대한 일반화 성능을 보장하지는 않는다.

따라서 최종 검증 전략은 2022년 holdout을 중심으로 하되, rolling validation 결과를 함께 참고하는 방식이 적절하다.

## 2.22 검증데이터 구축 방안 최종 정리

이 대회의 최종 검증 전략은 다음과 같이 설정한다.

### 1. 기본 검증: Time-based Holdout

가장 중요한 검증 기준은 다음 구조이다.

| 구분 | 기간 |
|---|---|
| 학습 데이터 | 2019~2021년 |
| 검증 데이터 | 2022년 |

이 방식을 기본 검증으로 사용하는 이유는 test 데이터가 2024년이기 때문이다. 즉, 실제 대회 상황은 과거 데이터로 미래 시점의 수익률을 예측하는 구조이다. 따라서 랜덤 분할보다 시간 순서를 반영한 2022년 holdout이 더 적절하다.

### 2. 보조 검증: Rolling Validation

단일 연도 검증의 한계를 보완하기 위해 rolling validation도 함께 확인한다.

| 학습 구간 | 검증 구간 | 의미 |
|---|---|---|
| 2019 | 2020 | 강한 상승장에 대한 성능 확인 |
| 2019~2020 | 2021 | 부진한 시장에 대한 성능 확인 |
| 2019~2021 | 2022 | test에 가장 가까운 시점의 성능 확인 |

rolling validation은 모델이 특정 시장 국면에만 잘 맞는지 확인하는 데 필요하다. 특히 이 데이터에서는 2020년과 2021년의 target 분포가 크게 다르기 때문에, 여러 연도에서 안정적인 전략을 찾는 것이 중요하다.

### 3. 랜덤 검증은 보조 지표로만 사용

랜덤 split은 빠르게 baseline을 확인하는 용도로는 사용할 수 있다. 그러나 최종 모델 선택 기준으로 사용하기에는 한계가 있다. 랜덤 split은 서로 다른 시장 국면과 같은 ticker의 여러 시점을 섞기 때문에 실제 2024년 test 상황보다 쉬운 검증이 될 수 있다.

### 4. ticker 기준 검증의 한계

ticker overlap 분석 결과, validation ticker의 대부분이 이전 학습 기간에도 존재한다. 따라서 Time-based Holdout은 시간 일반화 성능을 평가하는 데는 유용하지만, 완전히 새로운 기업에 대한 일반화 성능을 평가하는 데는 한계가 있다.

test의 ticker는 익명화되어 있으므로 ticker 이름 자체를 모델에 사용하는 전략은 적절하지 않다. 모델은 ticker를 암기하는 대신 재무지표, 섹터, 결측 패턴, valuation 상대값을 학습해야 한다.

### 최종 결론

이 대회에서는 다음 검증 원칙을 따른다.

1. 최종 모델 선택은 2022년 holdout RMSE를 중심으로 한다.
2. rolling validation으로 연도별 안정성을 확인한다.
3. 랜덤 split 점수는 참고용으로만 사용한다.
4. target과 prediction clipping은 validation 기준으로 범위를 선택한다.
5. leaderboard 점수와 validation 점수가 다르면 train/test 분포 차이와 시장 국면 차이를 원인으로 분석한다.

결론적으로 이 대회의 검증 전략은 **시간 기반 검증 + rolling validation + 예측값 분포 점검**을 함께 사용하는 방식이 가장 적절하다.

# 3. 모델의 개선 전략

앞선 데이터 분석 결과, 이 대회는 단순히 복잡한 모델을 적용한다고 성능이 좋아지는 문제가 아니다. `return_pct`는 극단값이 많은 fat-tailed target이고, feature에는 결측치와 이상치가 많으며, train과 test는 시간적으로 분리되어 있다.

따라서 모델 개선 전략은 다음 네 가지 관점에서 수립한다.

1. 공개 코드 중 참고할 만한 전략 분석
2. 내부 validation과 leaderboard 결과 차이 분석
3. 데이터 기반 개선 전략
4. 모델 기반 개선 전략

이 섹션에서는 실제 모델을 새로 학습하기보다, 앞선 EDA 결과를 바탕으로 어떤 방향으로 모델을 개선해야 하는지 정리한다.

## 3.1 공개 코드 중 가장 내용이 적합한 노트북의 전략 분석

> **AI 활용 질문**: 어떤 공개코드를 선택해야 내 현재 상황에 적합할까?
>
> **이 질문을 사용한 이유**: test ticker가 익명화되어 있기 때문에 종목명을 외우는 방식보다, 여러 기업의 펀더멘털을 상대 비교하는 전략이 적절한지 확인하기 위해 사용했다.

분석 대상으로 선택한 공개 노트북은 Kaggle의 **`Cross-Sectional ML for 1Y Stock Return Forecasting`** 이다.

이 노트북을 선택한 이유는 제목에서 드러나듯이 이 대회를 단순 시계열 예측이 아니라 **cross-sectional stock return forecasting** 문제로 접근하기 때문이다. 이 대회의 test ticker는 익명화되어 있고, 특정 종목명을 외워서 예측할 수 없다. 따라서 모델은 개별 기업명을 암기하는 것이 아니라, 같은 시점의 여러 기업을 재무지표 기준으로 비교하여 상대적으로 좋은 수익률을 낼 가능성이 있는 기업을 찾는 방향으로 학습해야 한다.

즉, 이 대회는 다음과 같이 이해하는 것이 적절하다.

> 특정 종목 하나의 가격 흐름을 예측하는 문제가 아니라, 여러 기업의 펀더멘털을 비교하여 1년 후 수익률이 상대적으로 높거나 낮을 기업을 구분하는 횡단면 예측 문제이다.

이 관점은 앞선 데이터 분석 결과와도 잘 맞는다. 우리는 섹터별 수익률 차이, valuation 지표의 극단값, 성장률과 수익성 지표의 왜도, train/test 분포 차이를 확인했다. 따라서 단순히 모든 기업을 하나의 절대 기준으로 비교하는 것보다, 섹터와 기업 특성을 고려한 상대 비교 전략이 더 타당하다.

분석 대상 공개 노트북:

- 노트북명: Cross-Sectional ML for 1Y Stock Return Forecasting(https://www.kaggle.com/code/avikdas567/cross-sectional-ml-for-1y-stock-return-forecasting)
- 작성자: avikdas567
- 플랫폼: Kaggle
- 사용 데이터: Predict 1-Year US Stock Returns from Fundamentals
- 선택 이유: 이 대회를 횡단면 주식 수익률 예측 문제로 해석하는 접근이 과제의 데이터 특성과 가장 잘 맞기 때문

### 3.1.1 공개 노트북 전략의 핵심 해석

이 노트북에서 참고할 핵심 전략은 “주식 수익률 예측을 종목별 시계열 문제가 아니라 횡단면 비교 문제로 본다”는 점이다.

일반적인 주가 예측은 특정 종목의 과거 가격 흐름을 보고 다음 가격을 예측하는 방식으로 접근하는 경우가 많다. 그러나 이 대회의 feature는 가격 시계열이 아니라 SEC 공시에서 추출한 재무·펀더멘털 지표이다. 또한 test의 ticker가 익명화되어 있으므로 종목명 자체를 사용하는 전략은 일반화 가능성이 낮다.

따라서 더 적절한 접근은 다음과 같다.

| 관점 | 잘못된 접근 | 더 적절한 접근 |
|---|---|---|
| ticker 사용 | 특정 종목명을 외움 | ticker는 제외하거나 제한적으로 사용 |
| feature 해석 | 모든 기업을 같은 기준으로 비교 | 섹터 내 상대 비교 사용 |
| valuation | P/E, P/B 원본값 그대로 사용 | clipping 후 섹터 내 순위화 |
| target | 극단 수익률을 그대로 맞추려 함 | target clipping과 prediction clipping 검토 |
| 검증 | 랜덤 split만 사용 | 시간 기반 validation 사용 |
| 모델 | 단일 모델에 의존 | 여러 모델 또는 여러 seed의 ensemble 사용 |

이 전략은 앞선 EDA 결과와 직접 연결된다. `pe_ttm`, `price_to_book`, `roe`, `debt_to_equity`는 극단값이 심했고, 섹터별 수익률 중앙값도 크게 달랐다. 따라서 단순 원본 feature보다 섹터 내 상대 순위, clipping, 결측 indicator를 포함한 feature engineering이 더 적절하다.

### 3.1.2 공개 노트북 전략에서 우리 분석에 반영할 점

공개 노트북의 접근에서 우리 노트북에 반영할 점은 다음과 같다.

첫째, 이 문제를 cross-sectional prediction으로 해석한다. test ticker가 익명화되어 있으므로 개별 종목명을 외우는 방식은 사용할 수 없다. 대신 각 기업의 valuation, profitability, growth, leverage, sector 정보를 이용해 상대적으로 좋은 기업과 나쁜 기업을 구분해야 한다.

둘째, 섹터 기준의 상대 비교가 필요하다. 같은 P/E 30이라도 Technology에서는 성장 기대를 반영한 값일 수 있지만, Utilities나 Real Estate에서는 과대평가 신호일 수 있다. 따라서 섹터 내 P/E 순위, 섹터 내 ROE 순위, 섹터 내 성장률 순위 같은 feature가 유효할 수 있다.

셋째, 극단값 관리가 중요하다. target인 `return_pct`뿐 아니라 feature에도 비정상적인 극단값이 많았다. 따라서 target clipping, feature clipping, prediction clipping을 모두 검토해야 한다.

넷째, 시간 기반 검증이 필요하다. train은 2019~2022년, test는 2024년이므로 랜덤 split만으로는 실제 test 상황을 제대로 반영하기 어렵다. 최종 모델 선택은 2022년 holdout과 rolling validation을 기준으로 해야 한다.

다섯째, 단일 모델보다 안정적인 ensemble이 적합하다. 금융 데이터는 신호보다 잡음이 큰 경우가 많으므로, 하나의 모델이 특정 연도나 극단값에 과적합될 위험이 있다. 여러 모델, 여러 seed, 여러 clipping 범위의 예측을 평균하면 예측 분산을 줄일 수 있다.

## 3.2 검증 데이터와 리더보드 결과의 차이 분석

이 대회에서는 validation 점수와 leaderboard 점수가 다를 가능성이 높다. 그 이유는 train, validation, test가 같은 분포에서 랜덤하게 나뉜 데이터가 아니기 때문이다.

학습 데이터는 2019~2022년이고, test는 2024년이다. 앞선 분석에서 연도별 target 분포가 크게 다르다는 점을 확인했다. 예를 들어 2020년은 평균 수익률과 중앙값이 매우 높은 상승장 성격을 보였고, 2021년은 평균과 중앙값이 음수인 부진한 구간이었다. 2022년은 상대적으로 안정적이었지만 여전히 극단값이 존재했다.

따라서 validation과 leaderboard의 차이는 다음 원인으로 발생할 수 있다.

| 원인 | 설명 |
|---|---|
| 시장 국면 차이 | 2024년 test가 2022년 validation과 다른 시장 환경일 수 있음 |
| target 분포 차이 | 상승장, 하락장, 횡보장에 따라 수익률 분포가 크게 달라짐 |
| feature drift | test에서 결측률과 일부 feature 분포가 train과 다름 |
| public/private split 차이 | public leaderboard 일부 데이터에 우연히 잘 맞을 수 있음 |
| 극단값 영향 | public 또는 private subset에 초극단 수익률 종목이 많으면 점수가 크게 흔들림 |
| 검증 방식 문제 | 랜덤 split은 실제 미래 예측 상황보다 쉬운 검증일 수 있음 |

따라서 leaderboard 점수는 단독으로 해석하면 안 된다. 내부 validation에서 좋아진 전략이 public leaderboard에서 나빠졌다면, 모델이 잘못되었다기보다 validation이 test 분포를 충분히 반영하지 못했을 가능성이 있다. 반대로 validation은 나빠졌는데 public leaderboard만 좋아진 경우에는 public leaderboard에 과적합되었을 가능성이 있다.


### 3.2.1 validation과 leaderboard를 함께 해석하는 기준

모델 개선 과정에서는 validation 점수와 leaderboard 점수를 다음 기준으로 함께 해석한다.

| 상황 | 해석 | 대응 |
|---|---|---|
| validation 개선 + public 개선 | 가장 신뢰할 수 있는 개선 | 해당 전략 유지 |
| validation 개선 + public 악화 | test 분포 차이 또는 validation 설계 문제 가능 | rolling validation과 예측 분포 재확인 |
| validation 악화 + public 개선 | public leaderboard 과적합 가능성 | private 성능 위험, 신중히 판단 |
| validation 비슷 + public 개선 | 예측 안정화 또는 우연 가능성 | 여러 제출 결과 비교 |
| validation 개선폭보다 public 개선폭이 과도하게 큼 | public subset에 우연히 맞았을 가능성 | final submission 선택 시 주의 |

이 대회에서 가장 신뢰할 수 있는 개선은 다음 조건을 동시에 만족하는 경우이다.

1. 2022년 holdout RMSE가 개선된다.
2. rolling validation에서 특정 연도에만 성능이 좋아지지 않는다.
3. 예측값 분포가 train target 분포와 비교해 지나치게 크거나 작지 않다.
4. public leaderboard 점수도 validation 방향과 일치한다.
5. clipping이나 ensemble 적용 후 성능이 안정된다.

따라서 최종 모델 선택은 public leaderboard 점수만이 아니라, 내부 validation과 public 점수의 일관성을 기준으로 해야 한다.

## 3.3 데이터 기반의 개선 전략

> **AI 활용 질문**: “이 데이터의 target 극단값, feature 결측, 재무비율 이상치를 고려하면 어떤 전처리 전략이 필요해?”
>
> **이 질문을 사용한 이유**: 앞선 EDA 결과를 실제 개선 전략으로 연결하기 위해 target clipping, feature clipping, log 변환, 결측 indicator를 정리하는 데 사용했다.

앞선 EDA를 바탕으로, 이 대회에서 가장 중요한 데이터 기반 개선 전략은 다음과 같다.

### 1. Target clipping

`return_pct`는 극단값이 매우 많다. 전체 target의 최댓값은 10,000%를 넘고, 99% 분위수도 약 299% 수준이었다. RMSE는 큰 오차에 민감하므로, 이런 초극단 수익률을 그대로 학습하면 모델이 일부 관측치에 과도하게 끌려갈 수 있다.

따라서 target clipping을 검토한다.

예시 전략:

| 전략 | 설명 |
|---|---|
| 1%~99% clipping | 가장 극단적인 target 양끝 1%를 제한 |
| 0.5%~99.5% clipping | 조금 더 완만한 clipping |
| 연도별 clipping | 각 연도 시장 국면 차이를 반영 |
| validation 기반 clipping 선택 | 2022 holdout RMSE가 가장 좋은 범위 선택 |

target clipping은 극단 수익률을 완전히 무시한다는 뜻이 아니다. 모델이 초극단값에 과도하게 맞춰 일반적인 관측치 예측을 망치는 것을 방지하기 위한 안정화 전략이다.

### 2. Feature clipping

feature에도 극단값이 매우 많았다. 특히 `pe_ttm`, `price_to_book`, `price_to_sales`, `roe`, `rote`, `debt_to_equity`, `revenue_growth_yoy`는 평균과 중앙값 차이가 크고, 최댓값이 비정상적으로 컸다.

따라서 비율형 feature에는 clipping을 적용한다.

| feature 유형 | 예시 | 처리 방향 |
|---|---|---|
| valuation | `pe_ttm`, `price_to_book`, `price_to_sales` | 1%~99% clipping |
| profitability | `roe`, `roa`, `rote`, `net_margin` | clipping |
| leverage | `debt_to_equity` | clipping |
| growth | `revenue_growth_yoy`, `revenue_growth_3y` | clipping |
| dividend | `dividend_yield` | clipping 또는 indicator화 |

이 전략은 모델이 비정상적인 재무비율에 과도하게 반응하지 않도록 돕는다. 특히 선형 모델에서는 clipping이 거의 필수이며, tree-based model에서도 예측 안정성을 높이는 데 도움이 된다.

### 3. 규모 변수 log 변환

`total_assets`, `revenue_ttm`, `market_cap`, `current_assets`, `long_term_debt`, `shares_outstanding` 같은 규모 변수는 소수의 초대형 기업 때문에 오른쪽 꼬리가 매우 길다. 이런 변수는 원본 스케일로 사용하면 대형 기업의 차이가 과도하게 반영될 수 있다.

따라서 규모 변수에는 `log1p` 변환을 적용한다.

예를 들어 매출이 10억 달러인 기업과 100억 달러인 기업의 차이는 중요하지만, 1,000억 달러와 1,100억 달러의 차이가 같은 방식으로 모델에 반영되는 것은 적절하지 않을 수 있다. log 변환은 규모의 상대적 차이를 더 안정적으로 표현한다.

처리 후보:

| 변수 | 의미 | 처리 방향 |
|---|---|---|
| `market_cap` | 시가총액 | log1p |
| `revenue_ttm` | 최근 12개월 매출 | log1p |
| `total_assets` | 총자산 | log1p |
| `current_assets` | 유동자산 | log1p |
| `current_liabilities` | 유동부채 | log1p |
| `long_term_debt` | 장기부채 | log1p |
| `shares_outstanding` | 발행주식수 | log1p |
| `shares_diluted` | 희석주식수 | log1p |

단, 음수 값이 존재하는 컬럼은 단순 `log1p`를 바로 적용하면 안 된다. 음수 가능성이 있는 경우에는 clipping 후 signed log 변환을 사용하거나, 해당 값을 별도 indicator로 처리해야 한다.

### 4. 결측 indicator 추가

결측치 분석에서 train과 test 모두 33개 컬럼에 결측이 존재했다. 특히 배당, 재고, 부채, 성장률, 수익성 관련 컬럼의 결측률이 높았다.

이 결측은 단순 오류가 아니라 기업 특성일 수 있다.

| 결측 컬럼 | 가능한 의미 |
|---|---|
| 배당 관련 컬럼 | 배당을 하지 않는 성장기업 |
| `inventory` | 재고가 없는 소프트웨어·금융·서비스 기업 |
| `pe_ttm` | 적자 기업 또는 이익이 매우 작은 기업 |
| `debt_to_equity` | 자본이 음수이거나 부채비율 계산이 불안정한 기업 |
| `gross_margin` | 업종 구조상 매출총이익률 해석이 어려운 기업 |

따라서 결측값을 중앙값으로 대체하는 것만으로는 부족하다. 각 feature에 대해 결측 여부를 나타내는 `_is_missing` 변수를 추가하면, 모델이 결측 자체의 정보를 활용할 수 있다.

예를 들어 `pe_ttm_is_missing = 1`인 기업은 P/E가 계산되지 않는 기업이라는 의미를 가질 수 있고, 이는 미래 수익률과 관련될 수 있다.

### 5. 섹터 내 상대 순위 feature

섹터 분석 결과, 업종별 수익률 중앙값과 재무지표 해석 기준이 다르게 나타났다. 따라서 전체 기업을 하나의 기준으로 비교하는 것보다, 섹터 내부에서 상대적으로 어떤 위치에 있는지를 표현하는 feature가 필요하다.

예시:

| 파생변수 | 의미 |
|---|---|
| `pe_ttm_sector_rank` | 같은 섹터 내 P/E 상대 순위 |
| `price_to_book_sector_rank` | 같은 섹터 내 P/B 상대 순위 |
| `roe_sector_rank` | 같은 섹터 내 ROE 상대 순위 |
| `revenue_growth_yoy_sector_rank` | 같은 섹터 내 최근 성장률 순위 |
| `debt_to_equity_sector_rank` | 같은 섹터 내 부채 부담 순위 |

이 방식은 “기술주 P/E 30”과 “유틸리티 P/E 30”을 같은 의미로 보지 않도록 도와준다. 또한 test ticker가 익명화되어 있어 종목명 암기가 불가능하므로, 섹터 내 상대 위치는 일반화 가능한 feature가 될 가능성이 높다.

### 6. 예측값 분포 관리

최종 제출 전에는 예측값의 분포를 반드시 확인해야 한다. RMSE는 큰 오차에 민감하기 때문에 일부 test row에서 비정상적으로 큰 예측값이 나오면 점수가 크게 악화될 수 있다.

확인할 항목은 다음과 같다.

| 항목 | 확인 이유 |
|---|---|
| 예측값 평균 | train target 평균과 지나치게 다른지 확인 |
| 예측값 중앙값 | 일반적인 예측 수준 확인 |
| 예측값 최솟값/최댓값 | 비정상적인 극단 예측 확인 |
| 1%, 99% 분위수 | 대부분 예측값의 범위 확인 |
| train target 분포와 비교 | 제출값이 현실적인 범위인지 확인 |

prediction clipping은 제출값의 안정성을 높이는 전략이다. 예를 들어 validation에서 예측값을 -80%~300% 범위로 제한했을 때 RMSE가 좋아진다면, test 예측에도 같은 범위를 적용할 수 있다.

단, clipping 범위는 임의로 정하지 않고 validation 결과를 기준으로 선택해야 한다.

## 3.4 모델 기반의 개선 전략

> **AI 활용 질문**: “결측과 이상치가 많은 tabular 금융 데이터에서 어떤 모델과 앙상블 전략이 적절해?”
>
> **이 질문을 사용한 이유**: 단일 선형 모델보다 tree-based boosting과 ensemble이 왜 적합한지 정리하고, 모델 기반 개선 방향을 세우기 위해 사용했다.

데이터 기반 전처리 이후에는 모델 구조를 개선해야 한다. 이 대회는 39개 내외의 정형 재무 feature를 사용하는 회귀 문제이므로, tabular data에 강한 모델을 중심으로 접근하는 것이 적절하다.

### 1. Baseline 모델

가장 먼저 단순 baseline을 만든다.

| 모델 | 목적 |
|---|---|
| 평균 예측 | 최소 기준선 |
| 중앙값 예측 | fat-tail target에서 안정적 기준선 |
| Ridge Regression | 선형 관계 확인 |
| RandomForest | 비선형 baseline 확인 |

baseline은 점수를 높이기 위한 모델이라기보다, 복잡한 모델이 정말 의미 있는 개선을 만드는지 판단하는 기준이다.

### 2. Tree-based boosting 모델

본격적인 성능 개선에는 LightGBM, XGBoost, CatBoost 같은 gradient boosting 계열 모델이 적합하다.

이유는 다음과 같다.

| 장점 | 설명 |
|---|---|
| 비선형 관계 학습 | 재무지표와 수익률은 단순 선형 관계가 아닐 가능성이 큼 |
| feature scaling에 덜 민감 | log/clipping 후 다양한 스케일 feature 처리 가능 |
| 결측값 처리에 강함 | 일부 모델은 결측 방향을 자체적으로 학습 가능 |
| feature interaction 학습 | valuation × growth, profitability × leverage 같은 조합 학습 가능 |
| feature importance 확인 | 어떤 재무지표가 중요한지 해석 가능 |

특히 이 대회에서는 `valuation`, `profitability`, `growth`, `leverage`, `sector` 사이의 상호작용이 중요할 수 있다. 예를 들어 저평가 기업이라도 성장성이 낮으면 수익률이 낮을 수 있고, 성장성이 높더라도 부채 부담이 크면 위험이 커질 수 있다. tree-based boosting은 이런 비선형 관계를 포착하는 데 유리하다.

### 3. Robust objective와 target 안정화

공식 평가 지표는 RMSE이므로 기본적으로 squared error 기반 학습이 평가 지표와 잘 맞는다. 그러나 target에 극단값이 많기 때문에 squared error만 사용하면 모델이 극단 수익률에 과도하게 끌려갈 수 있다.

따라서 다음 전략을 비교한다.

| 전략 | 목적 |
|---|---|
| squared error | 공식 RMSE와 직접 대응 |
| Huber loss | 극단값 영향 완화 |
| MAE 기반 비교 | 일반적인 관측치 예측 안정성 확인 |
| target clipping 후 RMSE | 극단값에 덜 끌리는 모델 학습 |
| prediction clipping | 제출값 안정화 |

최종 선택은 validation RMSE를 기준으로 하되, 특정 전략이 한 연도에서만 좋아지는지 rolling validation으로 함께 확인해야 한다.

### 4. Ensemble 전략

금융 데이터는 신호 대비 잡음이 크기 때문에 단일 모델에 의존하면 불안정할 수 있다. 따라서 ensemble은 이 대회에서 중요한 개선 전략이다.

가능한 ensemble 방식은 다음과 같다.

| Ensemble 방식 | 설명 |
|---|---|
| seed ensemble | 같은 모델을 여러 random seed로 학습 후 평균 |
| model ensemble | LightGBM, XGBoost, CatBoost 예측 평균 |
| clipping ensemble | target clipping 범위를 다르게 한 모델 평균 |
| validation weighted ensemble | validation RMSE가 낮은 모델에 더 큰 가중치 |
| rank-based ensemble | 예측값의 순위를 평균해 극단값 영향 완화 |

이 대회에서는 특히 `clipping ensemble`이 유용할 수 있다. target을 강하게 clipping한 모델은 안정적이지만 극단 상승 종목을 잘 못 맞출 수 있고, clipping을 약하게 한 모델은 극단값을 더 반영하지만 불안정할 수 있다. 두 모델을 평균하면 안정성과 극단값 대응의 균형을 잡을 수 있다.



### 5. 최종 모델 개선 방향

> **AI 활용 질문**: “Kaggle 제출 파일을 만들기 전에 예측값 분포에서 무엇을 확인해야 해?”
>
> **이 질문을 사용한 이유**: RMSE는 극단 예측값에 민감하므로, 제출 전에 예측값의 평균·중앙값·최솟값·최댓값·분위수를 확인하고 prediction clipping 여부를 결정하기 위해 사용했다.

이 대회에서 가장 적절한 최종 모델링 방향은 다음과 같다.

1. 기본 feature에서 ticker와 날짜 문자열은 제외한다.
2. `sector_code`는 유지한다.
3. 결측 indicator를 추가한다.
4. 비율형 feature는 clipping한다.
5. 규모형 feature는 log 변환한다.
6. 섹터 내 상대 순위 feature를 추가한다.
7. target clipping 범위를 validation으로 선택한다.
8. LightGBM, XGBoost, CatBoost 계열 모델을 비교한다.
9. 2022 holdout을 중심으로 모델을 선택한다.
10. rolling validation으로 연도별 안정성을 확인한다.
11. 최종 test 예측값은 prediction clipping으로 안정화한다.
12. 서로 다른 seed와 모델을 ensemble한다.

핵심은 복잡한 모델 하나를 만드는 것이 아니라, 데이터의 구조적 문제를 반영한 전처리와 검증 전략을 함께 사용하는 것이다.